# Escenario 6 — Extracción de Datos Estructurados

**Dominios:** Prompt Engineering & Structured Output · Context Management & Reliability

## Contexto
Construís un sistema que extrae información de documentos no estructurados (facturas, contratos, reportes), valida la salida usando JSON schemas y mantiene alta precisión. Debe manejar casos edge con elegancia.

## Lo que vas a implementar
1. Tool de extracción con JSON schema (required vs. nullable)
2. Few-shot examples para formatos de documentos variados
3. Loop de validación-retry con feedback específico
4. Campos de validación semántica (calculated_total vs. stated_total)
5. Estrategia de revisión humana con puntajes de confianza
6. Identificar cuándo el retry NO va a funcionar

In [ ]:
import anthropic
import json
from typing import Optional

client = anthropic.Anthropic()

## Paso 1 — Schema de extracción con campos nullable

**Concepto clave:** Los campos opcionales deben ser `["type", "null"]` para evitar que el modelo invente valores que no existen en el documento.

In [ ]:
EXTRACTION_TOOL = {
    "name": "extract_invoice",
    "description": "Extraé datos estructurados de una factura o documento de compra",
    "input_schema": {
        "type": "object",
        "properties": {
            # Campos REQUERIDOS — siempre deben estar
            "document_type": {
                "type": "string",
                "enum": ["invoice", "receipt", "purchase_order", "credit_note", "other"],
                "description": "Tipo de documento"
            },
            "document_type_detail": {
                "type": ["string", "null"],
                "description": "Si document_type es 'other', especificar el tipo exacto"
            },
            
            # Campos OPCIONALES — null si no están en el documento
            "invoice_number": {
                "type": ["string", "null"],
                "description": "Número de factura. Null si no figura."
            },
            "issue_date": {
                "type": ["string", "null"],
                "description": "Fecha de emisión en YYYY-MM-DD. Null si no figura."
            },
            "vendor_name": {
                "type": ["string", "null"],
                "description": "Nombre del proveedor"
            },
            "customer_name": {
                "type": ["string", "null"],
                "description": "Nombre del cliente/comprador"
            },
            "currency": {
                "type": ["string", "null"],
                "enum": ["USD", "EUR", "ARS", "BRL", "other", None]
            },
            "subtotal": {
                "type": ["number", "null"],
                "description": "Subtotal antes de impuestos"
            },
            "tax_amount": {
                "type": ["number", "null"],
                "description": "Monto de impuestos"
            },
            "total_amount": {
                "type": ["number", "null"],
                "description": "Total final declarado en el documento"
            },
            "calculated_total": {
                "type": ["number", "null"],
                "description": "Suma calculada de ítems de línea. Para validar vs total_amount."
            },
            "line_items": {
                "type": ["array", "null"],
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": ["number", "null"]},
                        "unit_price": {"type": ["number", "null"]},
                        "line_total": {"type": ["number", "null"]}
                    }
                }
            },
            "confidence_scores": {
                "type": "object",
                "description": "Confianza por campo (0.0 a 1.0) para enrutamiento de revisión",
                "properties": {
                    "total_amount": {"type": "number"},
                    "invoice_number": {"type": "number"},
                    "issue_date": {"type": "number"}
                }
            }
        },
        "required": ["document_type"]
    }
}

## Paso 2 — Few-shot examples para formatos variados

In [ ]:
EXTRACTION_SYSTEM = """
Extraés datos estructurados de documentos financieros.

REGLAS CRÍTICAS:
- Si un campo no figura en el documento → devolver null, NO inventar valores
- Normalizar fechas a YYYY-MM-DD
- Normalizar montos a números sin símbolos de moneda ni separadores de miles
- Calcular calculated_total sumando los totales de line_items
- Asignar confidence_scores honestos por campo

EJEMPLOS:

Documento 1 (factura con todo explícito):
  "FACTURA N° 0001-00123456
  Fecha: 15/01/2024
  Proveedor: Tech Solutions SRL
  Producto: Laptop x2 @ $800 c/u = $1.600
  Mouse x5 @ $20 c/u = $100
  Subtotal: $1.700  IVA 21%: $357  TOTAL: $2.057"
  
Extracción correcta:
  invoice_number: "0001-00123456"
  issue_date: "2024-01-15"
  vendor_name: "Tech Solutions SRL"
  total_amount: 2057.0
  calculated_total: 1700.0  ← suma de line items (sin impuestos)
  confidence_scores: {total_amount: 0.95, invoice_number: 0.99}

Documento 2 (ticket sin número de factura):
  "Recibo de pago — Supermercado ABC
  Artículos: ...
  Total: $45.30"
  
Extracción correcta:
  document_type: "receipt"
  invoice_number: null  ← no figura, no inventar
  issue_date: null      ← no figura
  total_amount: 45.30
  confidence_scores: {total_amount: 0.92, invoice_number: 1.0}  ← 1.0 = seguro de que es null

Documento 3 (monto ambiguo):
  "Total aprox. $1.500 (puede variar según tipo de cambio)"
  
Extracción correcta:
  total_amount: 1500.0
  confidence_scores: {total_amount: 0.45}  ← baja confianza por ambigüedad
"""

In [ ]:
def extract_document(document_text: str) -> dict:
    """Extrae datos de un documento usando tool_use con JSON schema."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system=EXTRACTION_SYSTEM,
        tools=[EXTRACTION_TOOL],
        tool_choice={"type": "tool", "name": "extract_invoice"},
        messages=[{"role": "user", "content": f"Extraé los datos de este documento:\n\n{document_text}"}]
    )
    
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    return {}


# Probar con distintos tipos de documentos
doc1 = """
FACTURA ELECTRÓNICA
Número: B-0002-00054321
Fecha emisión: 20 de enero de 2024
De: Soluciones Digitales S.A.
Para: Empresa Cliente S.R.L.

Descripción                    Cant.    P.Unit    Total
Licencia Software Pro          1        $500,00   $500,00
Soporte mensual                3        $150,00   $450,00
Capacitación (5 horas)         5        $80,00    $400,00

Subtotal: $1.350,00
IVA (21%): $283,50
TOTAL A PAGAR: $1.633,50
"""

print("Documento 1 — Factura completa:")
result1 = extract_document(doc1)
print(json.dumps(result1, indent=2, ensure_ascii=False))

## Paso 3 — Loop de validación y retry

In [ ]:
def validate_extraction(data: dict) -> list[str]:
    """Validación semántica — detecta errores que el schema no puede detectar."""
    errors = []
    
    # Validación semántica: ¿los totales cuadran?
    if data.get("calculated_total") and data.get("total_amount"):
        diff = abs(data["calculated_total"] - data["total_amount"])
        # Permitir diferencia por impuestos (si subtotal + tax = total)
        subtotal = data.get("subtotal", 0) or 0
        tax = data.get("tax_amount", 0) or 0
        expected = subtotal + tax
        
        if expected > 0 and abs(expected - data["total_amount"]) > 0.02:
            errors.append(
                f"Inconsistencia de totales: subtotal({subtotal}) + tax({tax}) = {expected:.2f} "
                f"pero total_amount = {data['total_amount']}"
            )
    
    # Validación de fecha: formato correcto
    if data.get("issue_date"):
        date = data["issue_date"]
        if not (len(date) == 10 and date[4] == '-' and date[7] == '-'):
            errors.append(f"Fecha '{date}' no está en formato YYYY-MM-DD")
    
    return errors


def extract_with_retry(document: str, max_retries: int = 2) -> tuple[dict, list[str]]:
    """Extrae con loop de retry. Retorna (extracción_final, errores_residuales)."""
    
    extraction = extract_document(document)
    errors = validate_extraction(extraction)
    
    if not errors:
        print("✅ Extracción válida en el primer intento")
        return extraction, []
    
    print(f"⚠️ {len(errors)} error(es) de validación: {errors}")
    
    for attempt in range(max_retries):
        print(f"🔄 Retry {attempt + 1}/{max_retries} con feedback específico...")
        
        # Retry con el error específico en el prompt
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=EXTRACTION_SYSTEM,
            tools=[EXTRACTION_TOOL],
            tool_choice={"type": "tool", "name": "extract_invoice"},
            messages=[
                {"role": "user", "content": f"Extraé datos de este documento:\n\n{document}"},
                {"role": "assistant", "content": [
                    {"type": "tool_use", "id": "retry_1", "name": "extract_invoice", "input": extraction}
                ]},
                {"role": "user", "content": [
                    {"type": "tool_result", "tool_use_id": "retry_1",
                     "content": f"Error de validación: {'; '.join(errors)}. Por favor corregí la extracción."}
                ]}
            ]
        )
        
        for block in response.content:
            if block.type == "tool_use":
                extraction = block.input
        
        errors = validate_extraction(extraction)
        if not errors:
            print(f"✅ Válido después del retry {attempt + 1}")
            return extraction, []
    
    print(f"❌ Errores residuales después de {max_retries} retries: {errors}")
    return extraction, errors


# Test del retry loop
doc_ambiguous = """
Factura ABC-001
Ítem 1: $100
Ítem 2: $200
Total: $350  ← inconsistente con la suma de ítems
"""

print("Documento con inconsistencia de totales:")
final, remaining_errors = extract_with_retry(doc_ambiguous)
print("\nResultado final:")
print(json.dumps(final, indent=2, ensure_ascii=False))

## Paso 4 — Cuándo el retry NO va a funcionar

In [ ]:
print("ANÁLISIS: ¿Cuándo el retry va a funcionar y cuándo no?")
print()

casos = [
    {
        "caso": "Fecha en formato incorrecto (15/01/2024 en vez de 2024-01-15)",
        "retry_funciona": True,
        "razon": "El modelo puede reformatear la fecha que claramente está en el documento"
    },
    {
        "caso": "El número de factura está en un adjunto PDF separado no provisto",
        "retry_funciona": False,
        "razon": "La información simplemente no está en el documento fuente. El retry va a alucinar."
    },
    {
        "caso": "Los subtotales de los ítems no suman el total (error estructural en el output)",
        "retry_funciona": True,
        "razon": "El modelo puede re-calcular el calculated_total correctamente"
    },
    {
        "caso": "El campo 'nombre del cliente' dice 'ver contrato adjunto' (no está en el doc)",
        "retry_funciona": False,
        "razon": "El dato existe en un documento externo no provisto. Sin el contrato, siempre va a fallar."
    },
    {
        "caso": "El monto tiene formato '$1.234,56' pero el campo espera number",
        "retry_funciona": True,
        "razon": "El modelo puede normalizar el formato a 1234.56"
    }
]

for caso in casos:
    emoji = "✅" if caso["retry_funciona"] else "❌"
    print(f"{emoji} {caso['caso']}")
    print(f"   → {caso['razon']}")
    print()

## Paso 5 — Revisión humana basada en confianza

In [ ]:
def route_for_human_review(extractions: list[dict]) -> dict:
    """Enruta extracciones a revisión humana basándose en puntajes de confianza."""
    
    auto_approved = []
    needs_review = []
    
    # Umbral calibrado (en producción se calibraría con un conjunto de validación)
    CONFIDENCE_THRESHOLD = 0.85
    
    for extraction in extractions:
        scores = extraction.get("confidence_scores", {})
        
        # Confianza mínima en los campos críticos
        critical_fields = ["total_amount", "invoice_number"]
        critical_scores = [scores.get(field, 0) for field in critical_fields if field in scores]
        
        min_confidence = min(critical_scores) if critical_scores else 0
        
        if min_confidence >= CONFIDENCE_THRESHOLD:
            auto_approved.append(extraction)
        else:
            extraction["review_reason"] = f"Confianza baja en campo crítico: {min_confidence:.2f}"
            needs_review.append(extraction)
    
    return {
        "auto_approved": len(auto_approved),
        "needs_review": len(needs_review),
        "review_queue": needs_review
    }


# Simular batch de extracciones
sample_extractions = [
    {"invoice_number": "INV-001", "total_amount": 1500.0, 
     "confidence_scores": {"total_amount": 0.98, "invoice_number": 0.99}},
    {"invoice_number": "INV-002", "total_amount": 750.0, 
     "confidence_scores": {"total_amount": 0.62, "invoice_number": 0.95}},  # total_amount baja confianza
    {"invoice_number": None, "total_amount": 200.0, 
     "confidence_scores": {"total_amount": 0.90, "invoice_number": 0.70}},  # número baja confianza
    {"invoice_number": "INV-004", "total_amount": 3200.0,
     "confidence_scores": {"total_amount": 0.92, "invoice_number": 0.97}}
]

routing = route_for_human_review(sample_extractions)
print(f"Auto-aprobadas: {routing['auto_approved']}")
print(f"Para revisión humana: {routing['needs_review']}")
print()
for item in routing["review_queue"]:
    print(f"  → {item.get('invoice_number', 'N/A')}: {item.get('review_reason')}")

print()
print("NOTA IMPORTANTE: Una precisión general del 95% puede enmascarar")
print("un rendimiento del 70% en un tipo específico de documento.")
print("Siempre analizar precisión por tipo de documento Y por campo.")

## Preguntas de Reflexión

1. ¿Qué tipos de errores elimina `tool_use` con JSON schema estricto? ¿Qué tipo de errores NO puede detectar?
2. ¿Por qué los campos opcionales deben ser `["type", "null"]` en lugar de simplemente omitirlos del schema?
3. Si el retry no funciona para un campo específico, ¿qué indica eso sobre el documento fuente?
4. ¿Qué es el `calculated_total` y para qué sirve además del `total_amount`?
5. ¿Por qué una precisión agregada del 97% puede ser engañosa? ¿Qué análisis adicional necesitás hacer?